# Encontro 4 · CrewAI do zero: agentes por papéis

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/se-ai-agents-engineering/ai-agents-engineering-v2/blob/main/encontro-04/04-crewai.ipynb)

No notebook 03, o agente da Aurora virou um grafo: você escreveu cada nó e cada aresta. O **CrewAI** parte de outra ideia: em vez de descrever **os passos**, você descreve **as pessoas**. Cada agente tem um papel, um objetivo e uma história; cada tarefa diz o que precisa ser feito e o que deve sair; e uma equipe (*crew*) executa as tarefas. O laço do agente, a escolha de ferramentas e a coordenação entre agentes ficam por conta do framework.

É o caminho mais curto até um protótipo multi-agente. A pergunta deste notebook é **o que vem junto com esse atalho**: o que o framework passa a decidir por você, e o que fica mais difícil de ver quando a resposta sai errada.

### O que você vai sair sabendo

- Montar um agente e uma tarefa em CrewAI, e ler o que o framework mostra da execução.
- Dar uma ferramenta a um agente, e reconhecer que a descrição dela continua sendo o contrato.
- Exigir uma saída estruturada, com o contrato de saída do Encontro 1.
- Encadear dois agentes num processo sequencial, e ver o handoff entre eles.
- Montar o roteador com especialistas do notebook 01 como uma equipe hierárquica, com um gerente criado pelo CrewAI.
- Comparar o resultado com o supervisor do notebook 01, na mesma régua.

### Mapa do notebook

| Parte | O que você constrói | Chamadas ao modelo |
|---|---|---|
| 0 · Ambiente | — | 0 |
| 1 · Um agente, uma tarefa | um redator que resume um trecho de política | 1 |
| 2 · Uma ferramenta | um atendente que busca na wiki | 2 a 4 |
| 3 · Saída estruturada | o mesmo atendente, com o contrato de saída, em 4 casos | 8 a 16 |
| 4 · Dois agentes em sequência | um pesquisador e um redator, com handoff | 3 a 6 |
| 5 · Equipe hierárquica | especialistas e um gerente, em 5 casos | 20 a 40 |
| 6 · Síntese | o que o framework fez, e o que continuou seu | 0 |

### Este notebook precisa de chave

Ao contrário dos notebooks 01 a 03, **este não usa a gravação**. O CrewAI chama o modelo por conta própria, por fora do nosso `comum/modelo.py`, então não há como reproduzir as respostas sem chamar a API. Com um modelo barato, o notebook inteiro custa centavos.

- **No Colab:** crie o Segredo `API_KEY` (o ícone de chave na barra lateral) e libere o acesso deste notebook. Por padrão, o notebook usa o OpenRouter e o mesmo modelo da gravação. Para outro provedor, crie também os Segredos `BASE_URL` e `MODELO`.
- **Na sua máquina:** o notebook usa o `.env` da raiz, como o resto do repositório.
- **Sem chave:** leia o notebook. As explicações se sustentam sozinhas, e as saídas salvas mostram o que aconteceu.

### Peças que você já conhece

A busca é a do Encontro 3, as regras de resposta são as do Encontro 1, e a correção dos casos é a mesma régua dos notebooks 01 a 03. A diferença é que, aqui, o prompt de cada agente é escrito pelo CrewAI a partir dos campos que você preenche.

## Etapa 0 · Ambiente

No Colab, a célula baixa o repositório e instala o CrewAI. A primeira busca com palavras novas baixa também o modelo de embeddings da Aula 3, o que leva mais um ou dois minutos.

In [7]:
# Etapa 0 · No Colab, baixa o repositório e instala o CrewAI (1 a 2 minutos).
# Na sua máquina, instale antes: pip install -r encontro-04/requirements-demo.txt
import os, sys, subprocess
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")     # o CrewAI não envia telemetria
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
if "google.colab" in sys.modules:
    if not os.path.exists("/content/ai-agents-engineering"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/se-ai-agents-engineering/ai-agents-engineering-v2",
                        "/content/ai-agents-engineering"], check=True)
    os.chdir("/content/ai-agents-engineering/encontro-04")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "crewai>=1.0,<2",
                    "sentence-transformers"], check=True)
    from google.colab import userdata
    for nome in ("API_KEY", "BASE_URL", "MODELO"):           # os Segredos do Colab, se existirem
        try:
            os.environ[nome] = userdata.get(nome)
        except Exception:
            pass
sys.path.insert(0, os.getcwd())

import json, time
import busca
import multiagente as ma
from comum import medidor, modelo
from comum.resultado import Resultado

ma.preparar("usar")          # a gravação só é usada na Parte 5, para comparar com o supervisor
casos = {c["id"]: c for c in ma.carregar_casos()}
print("Ambiente pronto.")

Replay 'usar': 333 respostas gravadas com gpt-4o-mini em 2026-10-12 15:36
Há chave configurada: o que não estiver gravado será chamado ao vivo, com custo.
Busca: 90 trechos da wiki da Aurora, espaços confluence-rh, wiki-beneficios, wiki-financeiro, wiki-rh, wiki-ti
Ambiente pronto.


A célula seguinte cria o `llm`: o modelo que todos os agentes vão usar. Ela também define duas funções auxiliares para mostrar o uso de tokens e o custo de cada execução, aos preços da gravação.

In [8]:
from crewai import LLM

if not (os.getenv("API_KEY") or os.getenv("ANTHROPIC_API_KEY")):
    raise RuntimeError(
        "Sem chave de API. Este notebook chama o modelo ao vivo.\n"
        "No Colab: crie o Segredo API_KEY (ícone de chave na barra lateral) e libere o acesso deste notebook.\n"
        "Sem chave, leia o notebook pelas saídas já salvas.")

if os.getenv("PROVEDOR") == "anthropic" and os.getenv("ANTHROPIC_API_KEY"):
    nome_modelo = os.getenv("MODELO") or modelo.MODELO
    llm = LLM(model=f"anthropic/{nome_modelo}", api_key=os.environ["ANTHROPIC_API_KEY"], temperature=0)
else:
    # Qualquer API compatível com a da OpenAI: o prefixo "openai/" diz isso, e o base_url diz qual.
    nome_modelo = os.getenv("MODELO") or modelo.estado_replay().get("modelo") or modelo.MODELO
    base_url = os.getenv("BASE_URL") or "https://openrouter.ai/api/v1"
    llm = LLM(model=f"openai/{nome_modelo}", base_url=base_url, api_key=os.environ["API_KEY"], temperature=0)
print("modelo:", nome_modelo)


def custo(uso) -> float:
    """Custo em dólares, aos preços da gravação dos notebooks 01 a 03."""
    return (uso.prompt_tokens * medidor.PRECO_ENTRADA + uso.completion_tokens * medidor.PRECO_SAIDA) / 1_000_000


def uso_de(saida) -> str:
    u = saida.token_usage
    return (f"{u.successful_requests} chamadas ao modelo · {u.prompt_tokens} tokens de entrada, "
            f"{u.completion_tokens} de saída · US$ {custo(u):.5f}")

modelo: gpt-4o-mini


## Parte 1 · Um agente, uma tarefa

**Produz:** `saida`, o resultado de uma equipe com um agente e uma tarefa.

O CrewAI tem três peças básicas:

| Peça | O que é | Campos principais |
|---|---|---|
| **Agent** | Quem faz o trabalho | `role` (papel), `goal` (objetivo), `backstory` (história), `llm`, `tools` |
| **Task** | O que precisa ser feito | `description`, `expected_output` (o que deve sair), `agent` (quem faz) |
| **Crew** | A equipe que executa as tarefas | `agents`, `tasks`, `process` |

A diferença para o que vocês escreveram até aqui: **não há prompt**. O CrewAI monta o prompt do agente a partir do papel, do objetivo, da história e da tarefa. É conveniente, e é também a primeira coisa que deixa de ser sua.

Os nomes entre chaves na descrição, como `{trecho}`, são preenchidos pelo `kickoff(inputs=...)`. O trecho vem da busca do Encontro 3.

Com `verbose=True`, o CrewAI mostra a execução: o agente que começou, a tarefa e a resposta final.

In [9]:
from crewai import Agent, Task, Crew

trecho = busca.buscar(casos["N2"]["pergunta"])[0]["trecho"]     # a política de férias

redator = Agent(
    role="Redator do atendimento interno da Aurora",
    goal="Explicar políticas internas em linguagem simples, para qualquer colaborador entender",
    backstory="Você trabalha no time de Pessoas & Cultura e escreve as respostas do atendimento.",
    llm=llm, verbose=True,
)
resumir = Task(
    description="Resuma em uma frase, para um colaborador, o trecho de política abaixo.\n\n{trecho}",
    expected_output="Uma frase curta, sem jargão.",
    agent=redator,
)
saida = Crew(agents=[redator], tasks=[resumir], verbose=True).kickoff(inputs={"trecho": trecho})
print("\nresposta:", saida.raw)
print(uso_de(saida))

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a0bad8e5-43ca-4b9a-9f15-44e0e40b7393                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Resuma em uma frase, para um colaborador, o trecho de política abaixo.                                   │
│                                                                                                                 │
│  Política de férias > 3. Como pedir                                                                             │
│  3.1 A solicitação é feita no Portal do Colaborador com pelo menos 30 dias de antecedência e precisa ser        │
│  aprovada pela liderança imediata.                                                                              │
│  ID: 154b087e-23e0-495d-8769-7fe639596735                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redator do atendimento interno da Aurora                                                                │
│                                                                                                                 │
│  Task: Resuma em uma frase, para um colaborador, o trecho de política abaixo.                                   │
│                                                                                                                 │
│  Política de férias > 3. Como pedir                                                                             │
│  3.1 A solicitação é feita no Portal do Colaborador com pelo menos 30 dias de antecedência e precisa ser        │
│  aprovada pela liderança imediata.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redator do atendimento interno da Aurora                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Para pedir férias, faça a solicitação no Portal do Colaborador com pelo menos 30 dias de antecedência e        │
│  aguarde a aprovação do seu líder.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Resuma em uma frase, para um colaborador, o trecho de política abaixo.                                   │
│                                                                                                                 │
│  Política de férias > 3. Como pedir                                                                             │
│  3.1 A solicitação é feita no Portal do Colaborador com pelo menos 30 dias de antecedência e precisa ser        │
│  aprovada pela liderança imediata.                                                                              │
│  Agent: Redator do atendimento interno da Aurora                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


resposta: Para pedir férias, faça a solicitação no Portal do Colaborador com pelo menos 30 dias de antecedência e aguarde a aprovação do seu líder.
1 chamadas ao modelo · 158 tokens de entrada, 31 de saída · US$ 0.00031


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a0bad8e5-43ca-4b9a-9f15-44e0e40b7393                                                                       │
│  Final Output: Para pedir férias, faça a solicitação no Portal do Colaborador com pelo menos 30 dias de         │
│  antecedência e aguarde a aprovação do seu líder.                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Como ler a saída

O painel do `verbose` mostra o agente, a tarefa e a resposta final. A última linha mostra quantas vezes o modelo foi chamado e quantos tokens leu. Repare no número de tokens de entrada: ele é bem maior que o trecho e as duas frases que você escreveu, porque inclui o prompt que o CrewAI montou em volta delas.

**Pergunta para a turma.** O papel, o objetivo e a história mudam o que o modelo responde? Que parte do prompt montado pelo CrewAI vocês gostariam de ler, e onde ela está?

## Parte 2 · Uma ferramenta

**Produz:** `atendente`, um agente com a busca da wiki.

### A ideia

Uma ferramenta, em CrewAI, é uma função Python com o decorador `@tool`. O nome vai no decorador, e **a docstring vira a descrição que o modelo lê**. É o contrato do Encontro 2, com outra roupa: se a docstring for vaga, o agente usa a ferramenta mal.

A busca é a mesma do `busca.py`, a configuração final do Encontro 3, na wiki inteira. É o agente único da Aula 3, agora em CrewAI.

O laço do agente (pensar, chamar a ferramenta, ler o resultado, decidir de novo) é do CrewAI: você não o escreve. O limite de voltas desse laço é o `max_iter`, o **critério de parada** do Encontro 1, agora num parâmetro do framework.

In [10]:
from crewai.tools import tool


@tool("buscar_politicas")
def buscar_politicas(pergunta: str) -> str:
    """Busca nas políticas oficiais e vigentes da Aurora: RH, TI, benefícios e financeiro.
    Devolve até 3 trechos, cada um com o id da página para citar como fonte.
    Use quando a resposta depende de uma regra da empresa."""
    return json.dumps([{k: r[k] for k in ("id", "titulo", "secao", "trecho")} for r in busca.buscar(pergunta)],
                      ensure_ascii=False)


def novo_atendente(verbose: bool = False) -> Agent:
    return Agent(
        role="Atendente do suporte interno da Aurora",
        goal="Responder dúvidas de colaboradores com base nas políticas oficiais, citando o id das páginas",
        backstory=ma.REGRAS,                       # as regras de resposta do Encontro 1
        tools=[buscar_politicas], llm=llm, verbose=verbose,
        max_iter=5,                                # o critério de parada
    )


atendente = novo_atendente(verbose=True)
responder = Task(
    description="Dúvida de um colaborador: {pergunta}",
    expected_output="A resposta em até três frases, com o id das páginas usadas entre colchetes.",
    agent=atendente,
)
saida = Crew(agents=[atendente], tasks=[responder], verbose=True).kickoff(
    inputs={"pergunta": casos["T1"]["pergunta"]})
print("\nresposta:", saida.raw)
print(uso_de(saida))

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9f48d626-f075-4f7c-bf31-eb51b863eab2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Dúvida de um colaborador: Estou com o erro 809 na VPN, o que eu faço?                                    │
│  ID: 528577bf-03b7-4b9d-b5ea-51f1abc5b063                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Atendente do suporte interno da Aurora                                                                  │
│                                                                                                                 │
│  Task: Dúvida de um colaborador: Estou com o erro 809 na VPN, o que eu faço?                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: buscar_politicas                                                                                         │
│  Args: {'pergunta': 'erro 809 na VPN'}                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Atendente do suporte interno da Aurora                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  A conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de       │
│  hotel e de aeroporto. Ative a opção "Modo TCP 443" no cliente de VPN e reconecte. Se não resolver, use o       │
│  roteamento pelo celular. Se o erro persistir, abra um chamado no portal de chamados de TI informando o código  │
│  [ti-vpn-codigos-de-erro].                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Dúvida de um colaborador: Estou com o erro 809 na VPN, o que eu faço?                                    │
│  Agent: Atendente do suporte interno da Aurora                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


resposta: A conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e de aeroporto. Ative a opção "Modo TCP 443" no cliente de VPN e reconecte. Se não resolver, use o roteamento pelo celular. Se o erro persistir, abra um chamado no portal de chamados de TI informando o código [ti-vpn-codigos-de-erro].
3 chamadas ao modelo · 1255 tokens de entrada, 144 de saída · US$ 0.00198


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9f48d626-f075-4f7c-bf31-eb51b863eab2                                                                       │
│  Final Output: A conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em  │
│  redes de hotel e de aeroporto. Ative a opção "Modo TCP 443" no cliente de VPN e reconecte. Se não resolver,    │
│  use o roteamento pelo celular. Se o erro persistir, abra um chamado no portal de chamados de TI informando o   │
│  código [ti-vpn-codigos-de-erro].                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Como ler a saída

No painel do `verbose`, procure o uso da ferramenta: o nome, a pergunta que o modelo escreveu para a busca e o que ela devolveu. É o mesmo que o trace do notebook 01 mostrava, agora no formato do CrewAI.

**Pergunta para a turma.** No notebook 03, o laço do agente era uma aresta que voltava, escrita por vocês. Aqui, onde está esse laço? O que vocês perderam e o que ganharam ao não escrevê-lo?

## Parte 3 · Saída estruturada: o contrato de saída

**Produz:** `estruturado`, o placar do atendente em quatro casos, com a régua dos notebooks anteriores.

### A ideia

Na Parte 2, a resposta era texto livre. Para corrigir os casos automaticamente, como nos notebooks 01 a 03, a resposta precisa seguir o **contrato de saída** do Encontro 1: desfecho, fontes e resposta. Em CrewAI, isso se pede com `output_pydantic`: você declara o formato numa classe Pydantic, e o CrewAI pede esse formato ao modelo e valida o que voltar.

A função `rodar_crew` roda uma equipe em vários casos e corrige cada um com a mesma régua de sempre (`ma.avaliar`). Ela é reaproveitada na Parte 5.

In [11]:
from typing import Literal
from pydantic import BaseModel


class RespostaAurora(BaseModel):
    desfecho: Literal["responder", "escalar", "nao_sei"]
    fontes: list[str]
    resposta: str


def rodar_crew(equipe: Crew, ids: list[str]) -> list[dict]:
    """Roda a equipe em cada caso e corrige com a régua dos notebooks 01 a 03."""
    linhas = []
    for cid in ids:
        caso = casos[cid]
        inicio = time.perf_counter()
        try:
            saida = equipe.kickoff(inputs={"pergunta": caso["pergunta"]})
            r = Resultado.de_dict(saida.pydantic.model_dump()) if saida.pydantic else Resultado.de_json(saida.raw)
            u = saida.token_usage
            chamadas, entrada, saida_tokens, gasto = (u.successful_requests, u.prompt_tokens,
                                                      u.completion_tokens, custo(u))
        except Exception as erro:
            r, chamadas, entrada, saida_tokens, gasto = Resultado("invalido", [], str(erro)[:200]), 0, 0, 0, 0.0
        ok, motivo = ma.avaliar(r, caso)
        linhas.append({"id": cid, "acertou": ok, "chamadas": chamadas, "tokens": entrada + saida_tokens,
                       "custo": gasto, "segundos": time.perf_counter() - inicio})
        print(f"{cid:<4}{'ok ' if ok else 'ERR'}  {chamadas:>2} chamadas  {entrada + saida_tokens:>6} tokens  "
              f"US$ {gasto:.5f}  {time.perf_counter() - inicio:5.1f}s  {motivo}")
    n = len(linhas)
    acertos = sum(l["acertou"] for l in linhas)
    print(f"\nacertos {acertos}/{n} · {sum(l['chamadas'] for l in linhas) / n:.1f} chamadas/caso · "
          f"{sum(l['tokens'] for l in linhas) / n:.0f} tokens/caso · US$ {sum(l['custo'] for l in linhas):.4f} no total")
    return linhas


responder_estruturado = Task(
    description="Dúvida de um colaborador: {pergunta}\n\n" + ma.REGRAS,
    expected_output="desfecho (responder, escalar ou nao_sei), fontes (ids das páginas) e uma resposta curta",
    agent=novo_atendente(), output_pydantic=RespostaAurora,
)
equipe_unica = Crew(agents=[responder_estruturado.agent], tasks=[responder_estruturado])
estruturado = rodar_crew(equipe_unica, ["N1", "E1", "S1", "X1"])

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

N1  ok    6 chamadas    4232 tokens  US$ 0.00524    3.6s  ok


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

E1  ok    9 chamadas   10402 tokens  US$ 0.01191    3.2s  ok
S1  ok   12 chamadas   19766 tokens  US$ 0.02159    2.9s  ok


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

X1  ok   13 chamadas   23726 tokens  US$ 0.02567    1.2s  ok

acertos 4/4 · 10.0 chamadas/caso · 14532 tokens/caso · US$ 0.0644 no total


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Os quatro casos cobrem uma pergunta normal (N1), uma exceção na política de viagens (E1), uma pergunta sem resposta na wiki (S1) e um pedido de elegibilidade, que deve ser escalado (X1).

**Pergunta para a turma.** Comparem com o agente único do notebook 01 nesses mesmos quatro casos: acertos, chamadas e tokens por caso. De onde vem a diferença de tokens?

## Parte 4 · Dois agentes em sequência: o handoff, de novo

**Produz:** `saida`, a resposta de uma dupla: um pesquisador e um redator.

### A ideia

Num processo **sequencial**, as tarefas rodam em ordem, cada uma com o seu agente. Uma tarefa recebe o resultado de outra pelo parâmetro `context`. Aqui, o pesquisador busca os trechos, e o redator escreve a resposta só com o que o pesquisador passou.

```
pergunta ──► pesquisador (com a busca) ──► texto com os trechos ──► redator (sem busca) ──► resposta
```

O que passa de uma tarefa para a outra é **o texto que o pesquisador escreveu**: um resumo, nas palavras dele. É o handoff da Etapa 5 do notebook 01, e o risco é o mesmo, a **diluição de contexto**. O caso é o E1, em que o detalhe que decide a resposta é uma exceção: o pernoite em Brasília muda o limite de R$ 110 para R$ 160.

In [12]:
from crewai import Process

pesquisador = Agent(
    role="Pesquisador de políticas da Aurora",
    goal="Encontrar nas políticas oficiais os trechos que respondem à dúvida, com o id de cada página",
    backstory="Você só pesquisa; quem responde ao colaborador é o redator.",
    tools=[buscar_politicas], llm=llm, max_iter=5,
)
redator_final = Agent(
    role="Redator do atendimento interno da Aurora",
    goal="Escrever a resposta final ao colaborador, usando só o que o pesquisador encontrou",
    backstory=ma.REGRAS, llm=llm,
)
pesquisar = Task(
    description="Encontre os trechos de política que respondem a esta dúvida: {pergunta}",
    expected_output="Os trechos relevantes, cada um com o id da página.",
    agent=pesquisador,
)
redigir = Task(
    description="Escreva a resposta ao colaborador para a dúvida: {pergunta}",
    expected_output="desfecho (responder, escalar ou nao_sei), fontes (ids das páginas) e uma resposta curta",
    agent=redator_final, context=[pesquisar], output_pydantic=RespostaAurora,
)
dupla = Crew(agents=[pesquisador, redator_final], tasks=[pesquisar, redigir], process=Process.sequential)

saida = dupla.kickoff(inputs={"pergunta": casos["E1"]["pergunta"]})
print("O que o pesquisador passou ao redator:\n")
print(saida.tasks_output[0].raw)
print("\nA resposta final:", saida.pydantic)
print(uso_de(saida))

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

O que o pesquisador passou ao redator:

Os trechos relevantes são:

1. "Viagens a trabalho e reembolso de despesas > 5. Alimentação  
5.1 As refeições durante a viagem são reembolsadas mediante nota fiscal, inclusive café da manhã quando não estiver incluído na diária do hotel.  
5.2 O limite diário de alimentação em viagens nacionais é de R$ 110,00 por pessoa, somando todas as refeições do dia. Bebidas alcoólicas não são reembolsadas em nenhuma hipótese, nem mesmo em refeições com clientes.  
5.3 Esse teto não vale quando o pernoite for em São Paulo, Rio de Janeiro ou Brasília, cidades classificadas como de custo elevado. Nelas o limite diário passa a ser de R$ 160,00, desde que a diária de hotel não inclua jantar." (ID da página: fin-viagens-e-reembolso)

2. "Adiantamento de viagem > 2. Valor  
2.1 O valor do adiantamento é calculado a partir dos limites de alimentação e transporte da política de viagens, multiplicados pelo número de dias." (ID da página: fin-adiantamento-de-viagem) 

**Pergunta para a turma.**

- O texto que o pesquisador passou adiante traz a exceção de Brasília? A resposta final saiu com R$ 160?
- O redator não tem a busca. Se o pesquisador cortou o detalhe, o redator tinha como saber?
- Em que situação essa divisão em dois papéis se pagaria? (Dica: pense no limite real do notebook 01. Algum dos dois tem fonte ou permissão própria?)

## Parte 5 · Equipe hierárquica: especialistas e um gerente

**Produz:** `hierarquica`, o placar da equipe hierárquica em cinco casos, e o supervisor gravado nos mesmos casos.

### A ideia

No processo **hierárquico**, as tarefas não têm agente fixo. O CrewAI cria um **gerente**, com prompt próprio, que lê a tarefa e decide a qual agente delegar, como o supervisor do notebook 01. A diferença é quem escreveu esse coordenador: lá, vocês; aqui, o framework.

Os especialistas são os do notebook 01: RH (com o Financeiro dentro), TI e Benefícios. Cada um tem a sua própria busca, presa aos espaços da sua área. **O controle de acesso por fonte continua do nosso lado, na ferramenta**, e não no framework.

Os cinco casos cobrem uma pergunta normal (N1), uma rota enganosa (C2), um termo exato (T1), um pedido de elegibilidade (X1) e uma pergunta de duas áreas (R1).

In [13]:
def ferramenta_de_busca(area: str, espacos: list[str]):
    @tool(f"buscar_politicas_{area}")
    def buscar(pergunta: str) -> str:
        """Busca nas políticas oficiais e vigentes da área. Devolve trechos com o id da página para citar."""
        return json.dumps([{k: r[k] for k in ("id", "titulo", "secao", "trecho")}
                           for r in busca.buscar(pergunta, espacos)], ensure_ascii=False)
    return buscar


especialistas = [
    Agent(role=f"Especialista de {area['titulo']}",
          goal=f"Responder dúvidas de {area['descricao']}, citando o id das páginas.",
          backstory=ma.REGRAS, tools=[ferramenta_de_busca(nome, area["espacos"])], llm=llm,
          allow_delegation=False, max_iter=5)
    for nome, area in ma.equipe(3).items() if area["espacos"]
]

tarefa = Task(
    description="Dúvida de um colaborador da Aurora: {pergunta}\n\n" + ma.REGRAS,
    expected_output="desfecho (responder, escalar ou nao_sei), fontes (ids das páginas) e resposta curta",
    output_pydantic=RespostaAurora,
)
equipe = Crew(agents=especialistas, tasks=[tarefa], process=Process.hierarchical, manager_llm=llm)
print(len(especialistas), "especialistas e um gerente criado pelo próprio CrewAI\n")

CASOS_DEMO = ["N1", "C2", "T1", "X1", "R1"]
hierarquica = rodar_crew(equipe, CASOS_DEMO)

3 especialistas e um gerente criado pelo próprio CrewAI



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

N1  ERR  68 chamadas  107640 tokens  US$ 0.12234    1.1s  desfecho 'escalar', esperado 'responder'


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C2  ERR   0 chamadas       0 tokens  US$ 0.00000    0.0s  saída fora do contrato


...

T1  ERR  72 chamadas  113916 tokens  US$ 0.12936    1.1s  desfecho 'nao_sei', esperado 'responder'


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

X1  ok   76 chamadas  122664 tokens  US$ 0.13871    0.9s  ok


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

R1  ERR  80 chamadas  134080 tokens  US$ 0.15115    1.0s  desfecho 'nao_sei', esperado 'responder'

acertos 1/5 · 59.2 chamadas/caso · 95660 tokens/caso · US$ 0.5416 no total


E o supervisor do notebook 01, nos mesmos cinco casos, a partir da gravação:

In [14]:
sup = ma.rodar(ma.supervisor(3), [casos[c] for c in CASOS_DEMO], mostrar=False)
ma.placar(sup)

arquitetura         acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
supervisor · 3 esp  5/5      4/5         3.8            3406         0.0224             0.00447             4.7     


### Como comparar

Olhem primeiro **chamadas e tokens por caso**: é ali que aparece o prompt do gerente, que vocês não escreveram e que viaja em toda chamada. Depois, o acerto.

Dois cuidados antes de concluir. São só cinco casos. E o CrewAI rodou ao vivo, enquanto o supervisor veio da gravação: rodando de novo, os dois dariam números um pouco diferentes. Uma diferença pequena pode ser só variância, e medir variância é o tema do Encontro 5.

**Pergunta para a turma.** Num caso que a equipe hierárquica errou, como vocês descobririam por quê? Comparem com o `mostrar_trace` do notebook 01.

> **Para ir além (opcional).** Rode a célula da equipe com `verbose=True` no `Crew` e procure as delegações do gerente. O que ele passou para cada especialista: a pergunta inteira ou um resumo?

## Parte 6 · Síntese: o que o framework fez, e o que continuou seu

### O mesmo atendimento, três jeitos

| Pergunta | Python à mão (01) | LangGraph (03) | CrewAI (este) |
|---|---|---|---|
| Quem escreve o prompt de cada agente? | nós | nós | o CrewAI, a partir de papel, objetivo e história |
| Quem escreve o coordenador? | nós | nós (as arestas) | o CrewAI (o gerente) |
| Dá para ver o que cada agente recebeu? | sim, com o `mostrar_trace` | sim, pelo estado de cada passo | com `verbose=True` ou uma ferramenta de tracing |
| Critério de parada | `MAX_PASSOS` | `recursion_limit` | `max_iter` |
| Retomar do meio e esperar uma pessoa | não | sim | parcial |
| Linhas de código de coordenação (aprox.) | 60 | 40 | 15 |
| Dá para gravar e reproduzir as respostas? | sim | sim | não: o modelo é chamado por fora do nosso código |

### O que continuou sendo de vocês, em qualquer framework

- **O contrato das ferramentas:** a descrição que o modelo lê, aqui na docstring.
- **A parede de acesso:** a busca de cada especialista, presa aos espaços da área.
- **O contrato de saída:** desfecho, fontes e resposta.
- **A régua:** os casos e a correção, que não mudaram de um notebook para o outro.

### Quando o CrewAI se paga

- **Se paga** num protótipo rápido, para mostrar uma ideia de equipe de agentes em poucos dias, quando o comportamento exato do coordenador importa menos que a velocidade.
- **Cobra caro** quando é preciso auditar cada decisão, controlar o prompt, retomar execuções ou reproduzir resultados. No caso da Aurora, a auditabilidade é um dos três problemas de arquitetura.

Nenhuma coluna da tabela é a certa. Cada uma troca controle por conveniência num ponto diferente. A pergunta de engenharia é qual dessas trocas o caso aguenta.

**Pergunta para a turma.**

Para a Aurora em produção: Python à mão, LangGraph ou CrewAI? E para um protótipo de duas semanas, para mostrar à diretoria? Respondam com um critério da tabela.